# Session 4: An Application to Drug Discovery - Translating Proteins back to DNA

So, we've seen how protein language models work, we know we can encode amino acid sequences and train a model to predict biophysical properties. But what about predicting another sequence?

In this notebook, we'll address the following question: given a sequence of amino acids, what is the corresponding DNA sequence which encodes that protein?

In autoregressive language models (like ChatGPT or Claude), we learn to predict each token in a sequence from a "vocabulary" of possible tokens. Here, we're going to do something similar - each amino acid can only be encoded by a set number of codons (triplets of nucleotide bases). For example, the amino acid proline is encoded by four possible codons: CCU, CCC, CCA, and CCG. In a way, this set of codons acts as the "vocabulary" for each amino acid, and our goal is to predict which codon should come next, given the full context.

What does this have to do with drug discovery? - Messenger RNA (mRNA) vaccines teach your body’s cells how to make a harmless piece of a protein from a virus. This triggers your immune system to produce antibodies, preparing your body to fight off the actual virus if you are exposed to it in the future. By learning the DNA sequence from the viral protein sequence, we could infer the mRNA sequence we need.

We're again going to use our pretrained protein language model (ESM2) to encode sequences and extract protein features. Then, we'll train a small "codon predictor" on top, which predicts which codon should come next for every amino acid.

**Note** that in this case we will be predicting codons **in parallel** rather than via an autoregressive process. This is because autoregressive models are computationally-heavy (often too expensive to train locally) and we know the exact length of the output sequence (one codon for every amino acid). The principles are the same.

In this case, we need paired DNA-amino acid sequences for training. We're going to use a dataset from [CodonTransformer](https://www.nature.com/articles/s41467-025-58588-7), which tries to tackle this same task. For the purpose of this workshop, we'll use a subset of the data which corresponds to *E. coli* proteins only.

Run the notebook one cell at a time and complete the activities to the best of your ability.

## 1. Setup

In this cell, we again load our libraries and fix a random seed. We're using mostly the same libraries, with the addition of:
- `time` — A library which records system time, we can use this to record how long downloads, training and inference take.
- `collections` — Provides some quality-of-life classes for working with dictionaries.
- `tqdm` — Adds a progress bar which we can use to track training progress.
- `pyarrow` and `fsspec` — These are just used to download and open the dataset.

In [ ]:
# These libraries are the same as before
import esm
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Library for recording system time
import time

# Counter tallies items, defaultdict fills in missing keys automatically (useful for working with dictionaries)
from collections import Counter, defaultdict

# Shows a nice progress bar which we can use to track training progress
from tqdm.auto import tqdm

# Used for downloading the data from the internet and reading its "Parquet" file format
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import fsspec

# Again, we fix a random seed for each library
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set the device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Configuration

This is where we define our fixed **hyperparameters** - these are the variables which control how our model will behave. You can find an explanation of each parameter in the cell below. The notebook has reasonable defaults, so try running the full notebook once, then come back and change them.


In [ ]:
# Data paths and other hyperparameters - these are fixed unless you move your data around
# Cache directory for holding pre-computed embeddings (so we don't have to re-compute on every run)
CACHE_DIR = Path('./esm_codon_cache'); CACHE_DIR.mkdir(exist_ok=True)
# Define data download URL - don't change this
PARQUET_URL  = 'https://huggingface.co/api/datasets/adibvafa/CodonTransformer/parquet/default/train/3.parquet'
# The CodonTransformer dataset contains a lot of data, so we'll only download the first 12 row groups and pull
# out the E. coli sequences. If you want more data, you can increase the value below (but this will take longer)
N_ROW_GROUPS = 12

# Sequence-length thresholds
# If the sequence is too short, we might not get enough context.
# Remember that self-attention is O(n^2) in the number of tokens, so the larger the sequence, the slower the model will run
# We'll only keep sequences between 30 and 250 amino acids for now
MIN_AA = 30
MAX_AA = 200

# Train-val-test split
# Fraction of proteins used to teach the model
TRAIN_FRAC = 0.90
# Fraction used to check progress during training
VAL_FRAC   = 0.05
# Fraction held back for the final fair evaluation
TEST_FRAC  = 0.05
# Sanity check that those three fractions actually sum to 100%
assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9

# Predictor architecture - this decodes ESM2 features into codons
# Predictor hidden dimension
D_MODEL = 256
# Number of self-attention heads in our transformer
NHEAD = 8
# Number of transformer-encoder layers
N_LAYERS = 4
# Dimension of MLP between self-attention layers
DIM_FF = 1024
# Dropout rate
DROPOUT = 0.1
# Positional encoding length
MAX_LEN = 1024

# Training hyperparameters
# How many proteins the model sees in one go
BATCH_SIZE = 32
# How many full passes over the training set
NUM_EPOCHS = 6
# Learning rate for the optimiser
LEARNING_RATE = 5e-4

## 3. Download the data

If you don't have access to the data already, you can download it directly using the cell below. This might take a while on first run, but afterwards we will skip it automatically.

In [ ]:
# Download the data - this isn't necessarily part of the workshop, but you can read on if you're interested.
# This line set up a filesystem that lets pyarrow read the Parquet file straight from a URL
hf_fs = pafs.PyFileSystem(pafs.FSSpecHandler(fsspec.filesystem('http')))
# Opens a handle to the Parquet file
pf = pq.ParquetFile(PARQUET_URL, filesystem=hf_fs)
# Record the start time so we can time the download
t0 = time.time()
# Pull just the columns we need from the first N_ROW_GROUPS chunks
df = pf.read_row_groups(list(range(N_ROW_GROUPS)), columns=['protein', 'dna', 'organism']).to_pandas()
# Keep only the rows that come from E. coli
df = df[df['organism'].str.contains('Escherichia coli', na=False)].reset_index(drop=True)
# Report how many rows we got and how long it took
print(f'Downloaded {len(df)} E. coli rows in {time.time()-t0:.1f}s.')

# Filter out sequences
# We remove any sequences which do not contain only the 20 standard amino acids (some data may contain "X" for unknown, 
# or non-canonical amino acids which ESM2 wasn't trained on)
# We also filter by min and max sequence length
# First, we define the 20 standard amino acid one-letter codes as a hard-coded string
STD_AA = 'ACDEFGHIKLMNPQRSTVWY'
# The allowed alphabet for protein strings is the standard amino acid list plus the stop symbol (underscore)
AA_ALPHABET = set(STD_AA + '_')
# Now we define a helper function that decides whether a row is valid
def is_okay(row):
    # Grab the protein and DNA strings from the row
    p, d = row['protein'], row['dna']
    # Length must be in range, DNA must be exactly 3x protein, and only allowed letters can appear
    return (MIN_AA <= len(p) <= MAX_AA and len(d) == 3*len(p) and not (set(p) - AA_ALPHABET) and not (set(d) - set('ACGT')))
# Apply the filter to every row
df = df[df.apply(is_okay, axis=1)].reset_index(drop=True)
# Make a new column with the stop symbol removed - this is the version we'll feed to ESM2
df['protein_no_stop'] = df['protein'].str.replace('_', '', regex=False)
print(f'After filtering: {len(df)} sequences.')

## 4. Tokenise input and output data

Here we assign the 22 possible tokens to our text strings: 20 standard amino acids, plus one pad (fills empty space in batches) and stop token (end of sequence).

We also assign 66 possible tokens to the codons: 64 (4^3) possible codons plus one pad and one "mask" token. The mask token acts as a placeholder for a possible codon, which the model will learn to translate into a real codon. This full set of tokens forms the "vocabulary" of our model.

We don't want our model to predict a completely incorrect codon for a given amino acid, so we'll also add a validity matrix - this is a 22 by 66 matrix where each element is True if the codon maps to the amino acid. We can use this to "hide" the impossible codons from our model so it doesn't accidentally assign them.

Note that we drop the stop tokens since ESM2 doesn't actually know what they are, we just used them to define the end of the sequence.

In [ ]:
# Define our amino acid vocabulary
# Special tokens: PAD fills empty space, MASK is what we feed in for "unknown" positions
PAD, MASK = '<pad>', '<mask>'
# Full list of amino acid tokens, starting with PAD
AA_TOKENS = [PAD] + list(STD_AA) + ['_']
# Create look-up dictionaries for mapping from amino acid string to integer and vice versa
aa2id = {t: i for i, t in enumerate(AA_TOKENS)}; PAD_S = aa2id[PAD]
id2aa = {i: t for t, i in aa2id.items()}

# Define our codon vocabulary
# This creates a list of all 64 possible three-letter DNA combinations
CODONS = [a+b+c for a in 'ACGT' for b in 'ACGT' for c in 'ACGT']
# Now we add pad and mask
CODON_TOKENS = [PAD, MASK] + CODONS
# Create look-up dictionaries for mapping from codon string to integer and vice versa
cd2id = {t: i for i, t in enumerate(CODON_TOKENS)}
id2cd = {i: t for t, i in cd2id.items()}

# Create some shortcuts for the special-token IDs
PAD_T, MASK_T = cd2id[PAD], cd2id[MASK]
# Get vocabulary sizes for the encoder input (amino acids) and predictor output (codons)
SRC_VOCAB, TGT_VOCAB = len(AA_TOKENS), len(CODON_TOKENS)

# Here, we define the complete codon to amino acid map - this shows the exact amino acid each codon encodes
TRANSL = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','AGT':'S','AGC':'S',
    'CCT':'P','CCC':'P','CCA':'P','CCG':'P','ACT':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCT':'A','GCC':'A','GCA':'A','GCG':'A','TAT':'Y','TAC':'Y',
    'TAA':'_','TAG':'_','TGA':'_',
    'CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','AAT':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAT':'D','GAC':'D','GAA':'E','GAG':'E','TGT':'C','TGC':'C','TGG':'W',
    'CGT':'R','CGC':'R','CGA':'R','CGG':'R','AGA':'R','AGG':'R',
    'GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

# Create the validity matrix
# Map each amino acid to the list of codon IDs that code for it (this basically creates the inverse of TRANSL above)
AA_TO_VALID = defaultdict(list)
for cd, aa in TRANSL.items(): AA_TO_VALID[aa].append(cd2id[cd])
# Create the Boolean validity matrix - VALID[a, c] is True if codon c codes for amino acid a
VALID = torch.zeros(SRC_VOCAB, TGT_VOCAB, dtype=torch.bool)
for aa, codon_ids in AA_TO_VALID.items():
    VALID[aa2id[aa], codon_ids] = True
# How many synonymous codons each amino acid has
n_syn = {aa: len(v) for aa, v in AA_TO_VALID.items()}

# Some helpful print statements
print('Synonymous codons per AA:', {a: n_syn[a] for a in 'ACDEFGHIKLMNPQRSTVWY'})
print(f'Validity mask shape {tuple(VALID.shape)}, {VALID.sum().item()} valid pairs.')

# Define functions to tokenise our input and output data on the fly
# Turn a protein string into a tensor of amino acid IDs
def tokenise_aa(p): return torch.tensor([aa2id[a] for a in p], dtype=torch.long)
# Turn a DNA string into a tensor of codon IDs (one ID per three letters)
def tokenise_dna(d): return torch.tensor([cd2id[d[i:i+3]] for i in range(0,len(d),3)], dtype=torch.long)

## 5. Load ESM2

We're using the same model as in `Session3_Protein_Language_Models.ipynb` (`esm2_t12_35M_UR50D`). This comprises 12 transformer layers, with a hidden dimension of 480, and 35 million parameters in total - small enough to run on CPU, but big enough that the representations carry meaningful information.

Again, we extract the features from the final layer and feed them into another, trainable model.

In [ ]:
# Set up the model and tokeniser
esm_model, esm_alphabet = esm.pretrained.esm2_t12_35M_UR50D()
# Set the model to evaluation mode and freeze the weights - remember, we don't want to train ESM2
esm_model.eval()
# Loop through every parameter and turn off gradient tracking
for p in esm_model.parameters(): p.requires_grad = False
# Move the model onto the chosen device
esm_model = esm_model.to(DEVICE)
# Use the tokeniser to convert sequences into integer IDs
batch_converter = esm_alphabet.get_batch_converter()
# Record the size of each per-residue feature vector ESM2 produces
ESM_DIM = esm_model.embed_dim
# Record the index of the final transformer layer (we'll read features from here)
ESM_LAYER = esm_model.num_layers

## 6. Pre-compute ESM2 features for every sequence

As in Session 3, we will cache our embeddings again. Since ESM2 is frozen, we only need to run each protein through it once, then we store the result for the next time we run the model.

You should find the encoded proteins already in the cache directory. But if you were to delete that data, running this cell would bring them back.

In [ ]:
# Setup the cache file
# Path that depends on the dataset size so different runs don't collide
CACHE_FILE = CACHE_DIR / f'esm_t12_35M_features_{len(df)}seqs.pt'

# Check whether we've saved features from a previous run
if CACHE_FILE.exists():
    # If the cache file exists, load the protein features from it
    esm_features = torch.load(CACHE_FILE, weights_only=True)
    print(f'Loaded {len(esm_features)} feature tensors.')
else:
    # If the cache file doesn't exist, calculate the protein features by encoding each sequence with ESM2
    esm_features = []
    # Track start time so we can report total processing time
    t0 = time.time()
    # Turn off autograd (we're not training)
    with torch.no_grad():
        # Loop over every protein with a progress bar
        for i, prot in enumerate(tqdm(df['protein_no_stop'].tolist(), desc='ESM forward', leave=False)):
            # Convert the protein into integer tokens
            _, _, tokens = batch_converter([('s', prot)])
            # Move the tokens onto the chosen device
            tokens = tokens.to(DEVICE)
            # Run ESM2 and ask for the final layer's activations
            out = esm_model(tokens, repr_layers=[ESM_LAYER], return_contacts=False)
            # Pull out the representation tensor
            rep = out['representations'][ESM_LAYER]
            # Get length of this protein
            L = len(prot)
            # Strip the start/end tokens and store as half-precision floats to save memory
            esm_features.append(rep[0, 1:L+1, :].cpu().half())
    # Report how long the whole pass took
    print(f'Processing time: {time.time()-t0:.1f}s')
    # Save the computed features so the next run can skip this step
    torch.save(esm_features, CACHE_FILE)

## 7. Build aligned training items

Now we bring it all together and set up our batches. Each batch needs a subset of the amino acid sequences (which may differ in length) and their corresponding DNA sequence (which also differ by the same length).

We'll set up a "collate" function, which PyTorch's `DataLoader` class takes as an argument by default. This explicitly defines how our data are separated into batches. In this case, we use it to convert our data from dictionary form, where sequences are of variable length, into tensors of fixed length. Since these tensors will be the same size, we can then stack them to create batches for training.

We also use the collate function to create a masked copy of the codon sequence (where all real codon positions are replaced with MASK tokens and the remaining space is padded). In practice, the sequence will look something like this: [MASK, MASK, MASK, ..., PAD, PAD]. This will be the sequence our predictor "sees" to get an idea of the tokens it needs to predict. The predictor then needs to work through each "MASK" token and determine which codon token should take its place.

In [ ]:
# Here, we create a dictionary with three tensors per amino acid sequence: the ESM2 embeddings, the amino acid tokens,
# and the codon tokens.
aligned = []
# Loop over every row in the table (every protein)
for i, row in df.iterrows():
    # Get protein string with the stop symbol removed
    p = row['protein_no_stop']
    # Length of this protein
    L = len(p)
    # Tokenise the full DNA sequence into codon IDs
    full_codons = tokenise_dna(row['dna'])
    # Bundle ESM features, amino acid IDs, and codon IDs into one dictionary
    aligned.append({
        'esm':    esm_features[i],
        'aa':     tokenise_aa(p),
        'codons': full_codons[:L],
    })

# Perform our train-val-test split
rng = np.random.default_rng(SEED); idx = rng.permutation(len(aligned))
n_train = int(TRAIN_FRAC * len(aligned)); n_val = int(VAL_FRAC * len(aligned))
train_data = [aligned[i] for i in idx[:n_train]]
val_data = [aligned[i] for i in idx[n_train:n_train+n_val]]
test_data = [aligned[i] for i in idx[n_train+n_val:]]
print(f'Train size: {len(train_data)}   Validation size: {len(val_data)}   Test size: {len(test_data)}')

# Define a custom dataset class (PyTorch expects this)
class CD(Dataset):
    # Store the list of items
    def __init__(self, items): self.items = items
    # Tell PyTorch how many items we have
    def __len__(self): return len(self.items)
    # Tell PyTorch how to fetch one item by index
    def __getitem__(self, i): return self.items[i]

# Here we define a "collate" function - This converts our data into batches by padding each sequence to the same length.
# It also creates a masked copy of the codon sequence (where all real codon positions are replaced with MASK tokens).
# We can feed this function as an argument to PyTorch's DataLoader to automatically batch the data according to our rules.
def collate_full_mask(batch):
    # Get the maximum length of the sequences in the batch
    Ls = [b['aa'].size(0) for b in batch]; L_max = max(Ls); B = len(batch)
    # Create tensors to hold the ESM features, amino acid tokens, and codon tokens. Fill empty space with pad tokens
    esm_t = torch.zeros(B, L_max, ESM_DIM, dtype=torch.float)
    aa_t = torch.full((B, L_max), PAD_S, dtype=torch.long)
    cd_t = torch.full((B, L_max), PAD_T, dtype=torch.long)

    # Fill the tensors with the data from the batch
    for i, item in enumerate(batch):
        # Length of this particular sequence
        L = item['aa'].size(0)
        # Copy the ESM features into the right slot
        esm_t[i, :L] = item['esm'].float()
        # Copy the amino acid tokens into the right slot
        aa_t[i, :L] = item['aa']
        # Copy the codon tokens into the right slot
        cd_t[i, :L] = item['codons']
    
    # Replace the real codon positions with mask tokens
    real = (cd_t != PAD_T)
    # Where the position is real, swap in MASK
    cd_in  = torch.where(real, torch.full_like(cd_t, MASK_T), cd_t)
    # Save the true codons for calculating the loss
    target = torch.where(real, cd_t, torch.full_like(cd_t, PAD_T))
    # Hand back the four tensors the model and loss function will use
    return esm_t, aa_t, cd_in, target

# Set up the data loaders
train_loader = DataLoader(CD(train_data), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_full_mask)
val_loader   = DataLoader(CD(val_data),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_full_mask)
print(f'Batches per epoch: {len(train_loader)}')

## 8. Set up the predictor

The predictor is a second network that we add on to the end of ESM2 (like our classifier MLP in the last notebook). This time, however, we're using a transformer-based architecture. This is much better for identifying local and global context in sequence data, which will help us more effectively predict codons in-sequence.

Remember that transformers don't account for the order of tokens unless you tell them to. The way we do this is by encoding the position of each token through a "position embedding".

A position embedding is added or concatenated to every token in the input so that the model can have some information about the position of the tokens with respect to each other. We could just number these tokens in order and take this as their positions (e.g. token 0 is labelled "0", token 1 is labelled "1", etc.) - this is easy for a human to understand, but not very useful for a neural network.

Typically, we use a sinusoidal positional embedding, which represents the position of each token as a vector of sine and cosine terms. For position $pos$ and embedding dimension index $i$, the sinusoidal position embedding is:

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{\frac{2i}{d}}}\right)
$$

$$
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{\frac{2i}{d}}}\right)
$$

where:

- $pos$ is the token position
- $i$ is the dimension index
- $d$ is the embedding dimension

For example, if we pick a simple case with $d = 2$, then the positional embedding of token 0 would be [$sin(0), cos(0)$] = [0, 1]. In this case, $d$ needs to be the same size as whatever we feed our predictor.

Why use a sinusoidal embedding? - Because sin and cosine are continuous, smooth, and easy to differentiate (during backpropagation). Also, it is practically impossible for any two integer positions to have the exact same sinusoidal embedding vector, which makes each position unique.

In [ ]:
# Define the sinusoidal position embedding, which encodes each token's position in the sequence
class SinusoidalPosEnc(nn.Module):
    # Pass in the important arguments the class needs - the dimension of our embedding and the max sequence length
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        # Take our token positions and encode them as a vector of sine and cosine terms
        # Here, we define an empty table that will hold one row per position
        pe = torch.zeros(max_len, d_model)
        # Create a vector of position indices (0, 1, 2, ...)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # Now we calculate the denominator
        div = 1.0 / (10000.0 ** (torch.arange(0, d_model, 2, dtype=torch.float) / d_model))
        # Then even dimensions get sines
        pe[:, 0::2] = torch.sin(pos * div)
        # And odd dimensions get cosines
        pe[:, 1::2] = torch.cos(pos * div)
        # Save the table as a buffer (this is a tensor which the model sees, but doesn't train)
        self.register_buffer('pe', pe.unsqueeze(0))
    # We add the position encoding directly to the token embeddings
    def forward(self, x): return x + self.pe[:, : x.size(1)]

## Define the predictor architecture
Now that we have our positional embedding, we can define our architecture. We'll use a fairly small architecture here, for two reasons:
1. ESM2 has done most of the hard work encoding protein features already.
2. We don't have time to train an enormous model.

The driving force behind this architecture is the `TransformerEncoderLayer` - this is a built-in transformer layer, supplied by PyTorch, which undertakes self-attention on our sequence data.

You'll notice that we embed our ESM2 features, our codon sequences and our token positions, then sum them together. At this point, the codon sequence is fully masked, so we gain no information about the codons themselves. By summing them, we encourage the transformer to learn to disentangle the embeddings directly. This may seem more complex, but transformers are actually very good at disentangling additive components. If you want to read more, check out the [original transformer paper](https://arxiv.org/abs/1706.03762).

In [ ]:
# Here we define the predictor architecture
class ESMCodonPredictor(nn.Module):
    # Pass the important arguments the class needs.
    def __init__(self, esm_dim, d_model=D_MODEL, nhead=NHEAD, n_layers=N_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT):
        super().__init__()
        # First we map the ESM2 features to the predictor's hidden size via a linear (fully-connected) layer
        # This maps ESM2's features to the same size as the positional embedding, d_model, so we can add them together
        self.esm_proj  = nn.Linear(esm_dim, d_model)
        # We use an embedding layer to convert codon tokens into vectors
        self.codon_emb = nn.Embedding(TGT_VOCAB, d_model, padding_idx=PAD_T)
        # Create the position embeddings
        self.pos = SinusoidalPosEnc(d_model)
        # Set up a transformer encoder layer (built into PyTorch already)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True, norm_first=True)
        # We repeat this layer n_layers times to get a full transformer encoder
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        # A final fully-connected layer to map the predictor's hidden size to the number of codons (i.e. to select codons
        # from our vocabulary)
        self.proj = nn.Linear(d_model, TGT_VOCAB)
        # Here we create a "buffer" (non-trainable tensor) to store the validity matrix, which we'll use to filter
        # out impossible codons during training
        self.register_buffer('valid', VALID)

    # Now we put it all together
    def forward(self, esm_feat, aa_ids, codon_ids, apply_constraint=False):
        # Get the pad token positions
        pad = (aa_ids == PAD_S)
        # Combine the ESM2 features, codon tokens and position embeddings
        # Remember, codon tokens are masked here, so this is just adding the same "unknown" information to every position
        h = self.esm_proj(esm_feat) + self.codon_emb(codon_ids)
        # Add the position encoding so the model knows where each token sits
        h = self.pos(h)
        # Pass the tokens through the transformer encoder
        h = self.encoder(h, src_key_padding_mask=pad)
        # Map the predictor's hidden size to the number of codons (i.e. to select codons from our vocabulary)
        logits = self.proj(h)
        # Apply the constraint mask
        if apply_constraint:
            # Look up the valid codons for each amino acid position
            v = self.valid[aa_ids]
            # Set the logits of invalid codons to negative infinity so they're never chosen
            logits = logits.masked_fill(~v, -float('inf'))
        # Return the codon scores
        return logits

# Define the model
model = ESMCodonPredictor(ESM_DIM).to(DEVICE)
# Print the parameter count
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.2f} million.')


## 9. Training Loop

Finally, we're ready to train.

**Activity:** Try training with a lower and higher value of `MAX_AA`. How does this effect training speed?

In [ ]:
# Set up our optimiser and loss
optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# Notice we again use a cross-entropy loss (ignoring the padding tokens), this is because we're essentially "classifying"
# each codon position
criterion = nn.CrossEntropyLoss(ignore_index=PAD_T)

# Define a helper function to get model loss and accuracy
def eval_loss(loader):
    # Switch the model to evaluation mode
    model.eval()
    # Track losses, c counts correct predictions and t counts total predictions
    losses=[]; c=t=0
    # Disable autograd
    with torch.no_grad():
        # Loop over every batch in the loader
        for esm_t, aa, cd_in, tgt in loader:
            # Move to device
            esm_t, aa, cd_in, tgt = (esm_t.to(DEVICE), aa.to(DEVICE), cd_in.to(DEVICE), tgt.to(DEVICE))
            # Get the model's codon scores
            logits = model(esm_t, aa, cd_in, apply_constraint=True)
            # Record the per-batch loss
            losses.append(criterion(logits.reshape(-1, TGT_VOCAB), tgt.reshape(-1)).item())
            # Take the most likely codon at each position
            preds = logits.argmax(-1); m = tgt != PAD_T
            # Count correct predictions (ignoring padding) and total positions
            c += ((preds==tgt) & m).sum().item(); t += m.sum().item()
    # Return mean loss and overall masked accuracy
    return float(np.mean(losses)), c/max(1,t)

# Set up our training loop
# Set up a dictionary that records loss/accuracy after each epoch
history = {'train_loss':[], 'val_loss':[], 'val_mask_acc':[]}
# Loop over the chosen number of epochs
for epoch in range(1, NUM_EPOCHS+1):
    # Switch the model to training mode and reset the loss list for this epoch
    model.train(); losses=[]
    # Create a progress bar over the training batches
    pbar = tqdm(train_loader, desc=f'epoch {epoch}/{NUM_EPOCHS}', leave=False)
    # Iterate over the training data
    for esm_t, aa, cd_in, tgt in pbar:
        # Move every tensor to device
        esm_t, aa, cd_in, tgt = (esm_t.to(DEVICE), aa.to(DEVICE), cd_in.to(DEVICE), tgt.to(DEVICE))
        # Get the model's predictions
        logits = model(esm_t, aa, cd_in, apply_constraint=True)
        # Compute the loss
        loss = criterion(logits.reshape(-1, TGT_VOCAB), tgt.reshape(-1))
        # Zero gradients, backpropagate, and update parameters
        optimiser.zero_grad(); loss.backward(); optimiser.step()
        # Record this batch's loss
        losses.append(loss.item())
        # Update the progress bar with a rolling mean of recent losses
        pbar.set_postfix(loss=f'{np.mean(losses[-30:]):.3f}')
    # Mean training loss for this epoch
    tl = float(np.mean(losses))
    # Evaluate on the validation set
    vl, va = eval_loss(val_loader)
    # Store epoch metrics in history
    history['train_loss'].append(tl); history['val_loss'].append(vl); history['val_mask_acc'].append(va)
    # Print a one-line summary for this epoch
    print(f'epoch {epoch:2d}: train {tl:.3f}  val {vl:.3f}  val_masked_acc {va:.3f}')

# Plot losses and model accuracy
# Set up two side-by-side panels
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
# Plot training and validation loss against epoch
axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['val_loss'], label='val')
# Label the axes
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('cross-entropy')
# Add legend, faint grid, and title
axes[0].legend(); axes[0].grid(True, alpha=.3); axes[0].set_title('Loss plots')
# Plot validation accuracy against epoch
axes[1].plot(history['val_mask_acc'], color='seagreen')
# Label the axes
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('validation set accuracy')
# Add a faint grid and title
axes[1].grid(True, alpha=.3); axes[1].set_title('Accuracy')
# Tidy spacing and show the figure
plt.tight_layout(); plt.show()


## 10. Example prediction

Here we'll input an example amino acid sequence, encode it using ESM2, and predict the corresponding codon sequence using our trained transformer. We'll predict in "parallel", meaning we predict the whole sequence in one pass.

**Activity:** Try picking a different amino acid from the test data set and compare the prediction to the ground truth.

In [ ]:
# Define a helper function to predict a single sequence
@torch.no_grad()
def predict_parallel(item):
    # Switch the model to evaluation mode
    model.eval()
    # Get ESM features and amino acids, define a masked codon sequence to add codons to. Move all onto the device
    esm_t = item['esm'].float().unsqueeze(0).to(DEVICE)
    aa = item['aa'].unsqueeze(0).to(DEVICE)
    cd = torch.full_like(aa, MASK_T)
    # Run the predictor with the valid-codon constraint switched on
    logits = model(esm_t, aa, cd, apply_constraint=True)
    # Pick the most likely codon at each position and convert to a plain list
    return logits.argmax(-1)[0].cpu().tolist()

# Take the first example from the test set
item = test_data[0]
# Predict the sequence
p_par = predict_parallel(item)
# Print the first 50 amino acids of this protein
print('Amino acids:', ''.join(id2aa[i] for i in item['aa'].tolist()[:50]) + '...')
# Print the first 50 codons of the true DNA sequence
print('Ground truth:', ''.join(id2cd[i] for i in item['codons'].tolist()[:50]) + '...')
# Print the first 50 codons predicted by our model
print('Prediction:  ', ''.join(id2cd[i] for i in p_par[:50]) + '...')
# Looks like they match up - let's quantify model performance

## 11. Evaluation

Now let's evaluate our model on the held-out test set. We'll compare the predictor against a baseline of picking the highest frequency codon (HFC) for each amino acid. This tells us if our model is at least as good as the most common guess.

In [ ]:
# Calculate the HFC baseline
# Define dictionary that counts codon usage for each amino acid in the training set
aa_to_codons = defaultdict(Counter)
# Walk through every training protein
for item in train_data:
    # Pair up each amino acid with its true codon
    for aa_id, cd_id in zip(item['aa'].tolist(), item['codons'].tolist()):
        # Skip padding and special tokens
        if cd_id >= 2 and aa_id != PAD_S:
            aa_to_codons[aa_id][cd_id] += 1
# For each amino acid, pick the codon that was used most often
hfc_codon = {aa: c.most_common(1)[0][0] for aa, c in aa_to_codons.items()}
# Define a helper HFC baseline function - this just predicts the most frequent codon for every amino acid
def hfc_pred(item): return [hfc_codon.get(a, PAD_T) for a in item['aa'].tolist()]

# Define a helper function to calculate codon accuracy
def codon_acc(true_ids, pred_ids):
    # Compare only up to the shorter of the two sequences
    # This only matters if apply_constraint = False, in which case the codon sequence could be of arbitrary length
    n = min(len(true_ids), len(pred_ids))
    # Get fraction of positions where prediction equals truth
    return sum(t==p for t,p in zip(true_ids[:n], pred_ids[:n])) / max(1, n)

# Get predictor and HFC predictions
par_preds = [predict_parallel(item) for item in tqdm(test_data, desc='parallel', leave=False)]
hfc_preds = [hfc_pred(item) for item in test_data]

# Calculate mean and standard deviation of codon accuracy
def mean_std(preds_list):
    # Per-protein accuracy for every prediction
    a = [codon_acc(it['codons'].tolist(), pr) for it, pr in zip(test_data, preds_list)]
    # Return the mean and standard deviation
    return float(np.mean(a)), float(np.std(a))
# Get mean and std for HFC and our predictor
h = mean_std(hfc_preds); pa = mean_std(par_preds)
print(f'Accuracy on test set ({len(test_data)} proteins):')
print(f'HFC baseline: {h[0]:.3f} +/- {h[1]:.3f}')
print(f'Our decoder: {pa[0]:.3f} +/- {pa[1]:.3f}')

# That's a pretty substantial improvement!

## 12. Diagnostic Plots

Here's a couple more plots to show us what the model is doing.

The first plot shows the accuracy of each individual amino acid, averaged over all sequences. We'll order the amino acids by codon degeneracy (i.e. amino acids with more codons come first on the horizontal axis). On the whole, we see that the more codons the model has to choose from, the lower the accuracy seems to be. But this could also be impacted by amino acid representation - the more common an amino acid is, the more data the model has to learn from.

The second plot shows the accuracy for each individual sequence, plotted as a histogram. Most of our sequences congregate towards the right (higher accuracy), but there's a small subpopulation centred near the HFC distribution. These might be protein families that were not present in ESM's training data.

In [ ]:
# Get the accuracy of each amino acid
# Define an empty dictionary to store our model and HFC accuracy for each amino acid
per_aa = defaultdict(lambda: {'hfc':[0,0], 'par':[0,0]})
# Loop over every test protein and its two sets of predictions
for item, hp, pp, in zip(test_data, hfc_preds, par_preds):
    # Loop over every position in the protein
    for i, aa_id in enumerate(item['aa'].tolist()):
        # Skip padding positions
        if aa_id == PAD_S: continue
        # Get the true codon at this position
        true_cd = item['codons'][i].item()
        # For both HFC and our model, update its counters
        for k, pr in [('hfc', hp), ('par', pp)]:
            # Count this as one more position seen
            per_aa[aa_id][k][1] += 1
            # If the prediction matches, count it as correct
            per_aa[aa_id][k][0] += int(pr[i] == true_cd)
# Build one row per amino acid for the summary table
rows = []
# Sort amino acids by how many synonymous codons they have (most first)
for aa_id in sorted(per_aa.keys(), key=lambda a: -n_syn.get(id2aa[a], 0)):
    # Set convenient short names for the HFC and parallel counters
    h_, p_ = per_aa[aa_id]['hfc'], per_aa[aa_id]['par']
    # Append the per-AA summary row
    rows.append({'aa': id2aa[aa_id], '#codons': n_syn[id2aa[aa_id]], 'n': h_[1], 'HFC': h_[0]/h_[1], 'parallel': p_[0]/p_[1]})
# Convert the rows to a DataFrame for easy printing/plotting
by_aa = pd.DataFrame(rows)
print(by_aa.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# Plot the accuracy of each amino acid
# Set up two side-by-side panels
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
# Grab the first panel
ax = axes[0]
# Pull out just the HFC and parallel columns indexed by amino acid
ax2 = by_aa.set_index('aa')[['HFC', 'parallel']]
# Plot grouped bars per amino acid
ax2.plot.bar(ax=ax, color=['lightgray', 'tab:orange', 'tab:blue'])
# Label the y-axis and set its range
ax.set_ylabel('codon accuracy'); ax.set_ylim(0, 1.05)
# Set the panel title
ax.set_title('Per-AA accuracy (AAs sorted by # synonymous codons)')
# Place the legend in the lower-right corner
ax.legend(loc='lower right', fontsize=8)

# Plot the accuracy of each sequence
# Per-sequence accuracy for our model
par_seq_acc = [codon_acc(it['codons'].tolist(), pr) for it, pr in zip(test_data, par_preds)]
# Per-sequence accuracy for the HFC baseline
hfc_seq_acc = [codon_acc(it['codons'].tolist(), pr) for it, pr in zip(test_data, hfc_preds)]
# Grab the second panel
ax = axes[1]
# Equally-spaced bins from 0.3 to 1.0
bins = np.linspace(0.3, 1.0, 30)
# Histogram of HFC per-sequence accuracies
ax.hist(hfc_seq_acc, bins=bins, alpha=0.55, label=f'HFC (mean {h[0]:.3f})', color='lightgray')
# Histogram of our model's per-sequence accuracies
ax.hist(par_seq_acc, bins=bins, alpha=0.55, label=f'ESM-predictor (mean {pa[0]:.3f})', color='tab:orange')
# Label the axes
ax.set_xlabel('per-sequence codon accuracy'); ax.set_ylabel('# test sequences')
# Add the legend and title
ax.legend(); ax.set_title('Distribution of per-sequence accuracy')
# Tidy spacing and show
plt.tight_layout(); plt.show()

## 13. Conclusion

To summarise, we've built a sequence-to-sequence model which predicts the specific DNA sequence that encodes a given amino acid sequence (with around 70-80% accuracy). This isn't too different from generative LLMs (like ChatGPT or Claude), in that we predict each token in a sequence, but note that our model is not autoregressive - our codon sequence is always the same size as our amino acid sequence (since we learn one codon for each amino acid). In autoregressive models, we would keep predicting tokens until a "stop" token is reached, but the principle is the same.

Our model gets pretty good accuracy already, but it could get even better with more data, parameters and training. By training a model like this to reconstruct (for example) DNA sequences for viral spike proteins, we could inform the design of mRNA vaccines.

## Final Activity
As always, see if you can push the test accuracy even higher. Try changing the hyperparameters and model architecture. If you're feeling brave, try adding another transformer block to the predictor. The person who gets the highest accuracy on the test set wins my undying respect.